# Практикум: matplotlib

Анализ и структуры данных, 9 класс. Визуализация данных.

Двенадцать задач на построение диаграмм: на наборе **Palmer Penguins** (`penguins.csv` лежит
рядом с этой тетрадью) и на небольших массивах, заданных прямо в коде.

Каждая функция строит рисунок и возвращает пару `(fig, ax)` — фигуру и область рисования.
Тесты проверяют не картинку, а объекты на ней: сколько линий и столбцов, какие подписи,
где проходят границы осей. Если все `assert` проходят, внизу печатается «ок»,
а под ячейкой появляется сам рисунок. Посмотрите на него обязательно: тест проверяет
устройство диаграммы, а читается она глазами.

**Как рисовать.** Рисунок создаётся строкой `fig, ax = plt.subplots()`, дальше всё
делается через `ax`: `ax.plot`, `ax.bar`, `ax.set_title`, `ax.set_xlabel`. Это объектный
стиль из раздела 3.3 лекции. Вызовы `plt.plot`, `plt.title` рисуют «куда-то в текущий
рисунок», и в функции, которую вызывают несколько раз, это быстро приводит к путанице —
некоторые тесты ловят именно её.

**Уровни.** Задачи 1–6 обязательные, в каждой одна новая идея. Задачи 7–10 сочетают
несколько идей. Задачи 11 и 12 со звёздочкой.

**Порядок работы.** Сначала выполните ячейку с загрузкой — она общая для всех задач.

### Загрузка данных

Выполните эту ячейку первой. В массивах `mass`, `flipper`, `bill` пропуски (`NaN`)
оставлены намеренно: с ними придётся разбираться.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("penguins.csv")
assert DATA.exists(), "Файл penguins.csv должен лежать рядом с этой тетрадью"
penguins = pd.read_csv(DATA)

SPECIES = ["Adelie", "Chinstrap", "Gentoo"]

# Столбцы как массивы NumPy, пропуски сохранены
mass = penguins["body_mass_g"].to_numpy()
flipper = penguins["flipper_length_mm"].to_numpy()
bill = penguins["bill_length_mm"].to_numpy()


def by_species(column):
    """Словарь: вид -> массив значений столбца для этого вида."""
    return {s: penguins.loc[penguins["species"] == s, column].to_numpy() for s in SPECIES}


mass_by_species = by_species("body_mass_g")

# Среднемесячная температура воздуха, °C.
# Округлённые многолетние значения, для упражнений.
MONTHS = ["янв", "фев", "мар", "апр", "май", "июн",
          "июл", "авг", "сен", "окт", "ноя", "дек"]
TEMP = {
    "Москва":   [-6.2, -5.9, -0.7, 6.8, 13.2, 17.0, 19.2, 17.0, 11.3, 5.6, -0.4, -4.3],
    "Сочи":     [6.5, 6.8, 9.0, 12.6, 16.8, 21.0, 24.1, 24.5, 20.6, 16.1, 11.4, 8.1],
    "Мурманск": [-10.1, -9.9, -6.0, -1.2, 4.0, 9.3, 12.8, 11.1, 7.0, 1.4, -4.3, -7.9],
}


def texts(items):
    """Тексты подписей списком: texts(ax.get_xticklabels()) -> ["a", "b", ...]"""
    return [t.get_text() for t in items]


plt.rcParams["figure.figsize"] = (7, 4)
print("версия matplotlib:", matplotlib.__version__)
print("размер таблицы:", penguins.shape)

## Часть 1. Обязательные задачи

По одной идее на задачу.

### Задача 1. Линейный график

Постройте ломаную по точкам `x`, `y` и отметьте сами точки кружками. Задайте заголовок
и подписи осей из аргументов.

```
fig, ax = line_chart(MONTHS, TEMP["Москва"], "Температура в Москве", "Месяц", "°C")
```

Функцию будут вызывать несколько раз подряд, и каждый вызов должен давать **новый**
рисунок, а не дорисовывать линию на старом.

*Подсказка:* форму точек задаёт параметр `marker`.

In [ ]:
def line_chart(x, y, title, xlabel, ylabel):
    """Ломаная с отмеченными точками, заголовком и подписями осей. Возвращает (fig, ax)."""
    fig, ax = plt.subplots()
    ax.plot(x, y, marker="o")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return fig, ax


fig, ax = line_chart(MONTHS, TEMP["Москва"], "Температура в Москве", "Месяц", "°C")
assert len(ax.lines) == 1, "на рисунке должна быть ровно одна линия"
line = ax.lines[0]
assert list(line.get_ydata()) == TEMP["Москва"]
assert line.get_marker() == "o", "точки нужно отметить кружками"
assert ax.get_title() == "Температура в Москве"
assert ax.get_xlabel() == "Месяц" and ax.get_ylabel() == "°C"
fig.canvas.draw()
assert texts(ax.get_xticklabels()) == MONTHS

fig2, ax2 = line_chart([1, 2, 3], [3, 1, 2], "t", "x", "y")
plt.close(fig2)
assert fig2 is not fig, "каждый вызов должен создавать новую фигуру"
assert len(ax2.lines) == 1
assert list(ax2.lines[0].get_xdata()) == [1, 2, 3]
assert len(ax.lines) == 1, "второй вызов дорисовал линию на первом рисунке"
print("Задача 1: ок")

### Задача 2. Столбцы по убыванию

Постройте столбчатую диаграмму по словарю «категория → число» так, чтобы столбцы шли
**по убыванию** высоты. Подпишите ось Y: `"Число наблюдений"`. Исходный словарь меняться
не должен.

```
counts = {"Torgersen": 52, "Biscoe": 168, "Dream": 124}
fig, ax = bar_sorted(counts)     # столбцы: Biscoe, Dream, Torgersen
```

*Подсказка:* `sorted(counts.items(), key=...)` даёт список пар в нужном порядке.
Столбцы на рисунке лежат в `ax.patches`, высота столбца — `get_height()`.

In [ ]:
def bar_sorted(counts: dict):
    """Столбчатая диаграмма, столбцы по убыванию высоты. Возвращает (fig, ax)."""
    pairs = sorted(counts.items(), key=lambda ij: -ij[1])
    labels = [i for i, _ in pairs]
    values = [j for _, j in pairs]
    fig, ax = plt.subplots()
    ax.bar(labels, values)
    ax.set_ylabel("Число наблюдений")
    return fig, ax


counts = {"Torgersen": 52, "Biscoe": 168, "Dream": 124}
backup = dict(counts)
fig, ax = bar_sorted(counts)
assert [p.get_height() for p in ax.patches] == [168, 124, 52]
fig.canvas.draw()
assert texts(ax.get_xticklabels()) == ["Biscoe", "Dream", "Torgersen"]
assert ax.get_ylabel() == "Число наблюдений"
assert counts == backup and list(counts) == list(backup), "исходный словарь изменился"

fig2, ax2 = bar_sorted({"a": 1, "b": 5, "c": 3})
fig2.canvas.draw()
plt.close(fig2)
assert [p.get_height() for p in ax2.patches] == [5, 3, 1]
assert texts(ax2.get_xticklabels()) == ["b", "c", "a"]

fig4, ax4 = bar_sorted({"x": 2, "y": 2, "z": 7})
plt.close(fig4)
assert [p.get_height() for p in ax4.patches] == [7, 2, 2]

fig3, ax3 = bar_sorted({})
plt.close(fig3)
assert len(ax3.patches) == 0
print("Задача 2: ок")

### Задача 3. Гистограмма с шагом

Постройте гистограмму, в которой все интервалы имеют ширину `step`, а их границы
кратны `step`. Первая граница — не больше минимума, последняя — строго больше максимума.
Пропуски в данных возможны.

```
fig, ax = hist_step(mass, 250)    # границы 2500, 2750, 3000, ..., 6500
```

*Подсказка:* параметр `bins` в `ax.hist` принимает не только число, но и **массив границ**.
Границы удобно строить через `np.arange`. Кратное `step` снизу даёт `np.floor(v / step) * step`.

In [ ]:
def hist_step(values, step):
    """Гистограмма с интервалами ширины step и границами, кратными step. Возвращает (fig, ax)."""
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    io = np.floor(values.min() / step) * step
    Hi = np.floor(values.max() / step) * step + step
    bins = np.arange(io, Hi + step, step)
    fig, ax = plt.subplots()
    ax.hist(values, bins=bins)
    return fig, ax

fig, ax = hist_step(mass, 250)
bars = ax.patches
assert all(np.isclose(p.get_width(), 250) for p in bars), "ширина всех интервалов должна быть 250"
assert np.isclose(bars[0].get_x(), 2500)
assert np.isclose(bars[-1].get_x() + bars[-1].get_width(), 6500)
assert sum(p.get_height() for p in bars) == 342      # 344 строки минус 2 пропуска

fig2, ax2 = hist_step(mass, 100)
plt.close(fig2)
assert np.isclose(ax2.patches[0].get_x(), 2700)
assert ax2.patches[-1].get_x() + ax2.patches[-1].get_width() > 6300
assert len(ax2.patches) == 37

fig3, ax3 = hist_step(np.array([5, 5, 5]), 10)
plt.close(fig3)
assert len(ax3.patches) == 1 and ax3.patches[0].get_height() == 3
assert np.isclose(ax3.patches[0].get_x(), 0)
print("Задача 3: ок")

### Задача 4. Диаграмма рассеяния без пропусков

Постройте диаграмму рассеяния по парам `(x[i], y[i])` с прозрачностью точек `alpha`.
Пары, где пропущено **хотя бы одно** из двух значений, на рисунок не попадают.

```
fig, ax = scatter_pairs(flipper, mass, 0.4)
```

*Подсказка:* маска «оба известны» — `~np.isnan(x) & ~np.isnan(y)`.
Точки диаграммы рассеяния лежат в `ax.collections[0]`, их координаты — `get_offsets()`.

In [ ]:
def scatter_pairs(x, y, alpha):
    """Диаграмма рассеяния по парам без пропусков. Возвращает (fig, ax)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    z = ~(np.isnan(x) | np.isnan(y))

    fig, ax = plt.subplots()
    ax.scatter(x[z], y[z], alpha=alpha)
    return fig, ax

fig, ax = scatter_pairs(flipper, mass, 0.4)
assert len(ax.collections) == 1
pts = np.asarray(ax.collections[0].get_offsets())
assert pts.shape == (342, 2)
assert not np.isnan(pts).any()
assert ax.collections[0].get_alpha() == 0.4
ax.set_xlabel("Длина ласта, мм")
ax.set_ylabel("Масса, г")

x = np.array([1.0, np.nan, 3.0, 4.0, 5.0])
y = np.array([10.0, 20.0, np.nan, 40.0, 50.0])
fig2, ax2 = scatter_pairs(x, y, 1)
plt.close(fig2)
pts2 = np.asarray(ax2.collections[0].get_offsets())
assert pts2.tolist() == [[1.0, 10.0], [4.0, 40.0], [5.0, 50.0]]
print("Задача 4: ок")

### Задача 5. Несколько рядов и легенда

Нарисуйте на одной области несколько ломаных: по одной на каждый ряд из словаря
«название → значения». У всех рядов общая ось `x`. Добавьте легенду; подписи в ней —
названия рядов в том же порядке, что в словаре.

```
fig, ax = multi_line(MONTHS, TEMP)
```

*Подсказка:* подпись для легенды задаётся параметром `label` при рисовании линии,
сама легенда — `ax.legend()`.

In [ ]:
def multi_line(x, series: dict):
    """По ломаной на каждый ряд, общая ось x, легенда. Возвращает (fig, ax)."""
    fig, ax = plt.subplots()
    for name, values in series.items():
        ax.plot(x, values, label=name)
    ax.legend()
    return fig, ax


fig, ax = multi_line(MONTHS, TEMP)
assert len(ax.lines) == 3
assert [l.get_label() for l in ax.lines] == ["Москва", "Сочи", "Мурманск"]
for line, values in zip(ax.lines, TEMP.values()):
    assert list(line.get_ydata()) == values
legend = ax.get_legend()
assert legend is not None, "нет легенды"
assert texts(legend.get_texts()) == ["Москва", "Сочи", "Мурманск"]
ax.axhline(0, color="0.7", linewidth=0.8)
ax.set_ylabel("°C")

fig2, ax2 = multi_line([0, 1], {"один": [5, 6]})
plt.close(fig2)
assert len(ax2.lines) == 1
assert texts(ax2.get_legend().get_texts()) == ["один"]
print("Задача 5: ок")

### Задача 6. Панели в ряд

По словарю «группа → значения» постройте фигуру из одного ряда панелей, по панели
на группу. На каждой панели — гистограмма с `bins` интервалами, заголовок панели —
название группы. Все панели используют **общие шкалы** по обеим осям, чтобы их можно было
сравнивать на глаз.

Верните фигуру и **список** областей рисования — в том числе когда группа одна.

```
fig, axes = hist_panels(mass_by_species, 10)
```

*Подсказка:* у `plt.subplots` есть параметры `sharex`, `sharey` и `squeeze`.
Прочитайте, что делает последний: `help(plt.subplots)`.

In [ ]:
def hist_panels(groups: dict, bins: int):
    """Гистограммы групп на панелях в один ряд, общие шкалы. Возвращает (fig, список axes)."""
    x = len(groups)
    fig, axes = plt.subplots(ncols=x, sharex=True, sharey=True, squeeze=False)
    axes = axes[0]
    for ax, (name, values) in zip(axes, groups.items()):
        values = np.asarray(values, dtype=float)
        values = values[~np.isnan(values)]
        ax.hist(values, bins=bins)
        ax.set_title(name)
    return fig, axes

fig, axes = hist_panels(mass_by_species, 10)
assert len(axes) == 3 and len(fig.axes) == 3
assert [a.get_title() for a in axes] == SPECIES
assert all(len(a.patches) == 10 for a in axes)
assert [sum(p.get_height() for p in a.patches) for a in axes] == [151, 68, 123]
assert axes[0].get_xlim() == axes[1].get_xlim() == axes[2].get_xlim(), "шкала X не общая"
assert axes[0].get_ylim() == axes[1].get_ylim() == axes[2].get_ylim(), "шкала Y не общая"
axes[0].set_ylabel("Число пингвинов")

fig2, axes2 = hist_panels({"только одна": np.array([1, 2, 2, 3])}, 3)
plt.close(fig2)
assert len(axes2) == 1
assert axes2[0].get_title() == "только одна"
print("Задача 6: ок")

## Часть 2. Сочетание идей

### Задача 7. Среднее и медиана на гистограмме

Постройте гистограмму с `bins` интервалами и проведите на ней две вертикальные линии:
среднее — сплошной линией с подписью `"среднее"`, медиану — пунктиром (`"--"`) с подписью
`"медиана"`. Добавьте легенду. Пропуски в данных возможны.

```
fig, ax = hist_center(mass, 20)
```

*Подсказка:* вертикальная линия через всю область — `ax.axvline(x, ...)`,
тип линии — параметр `linestyle`.

In [ ]:
def hist_center(values, bins: int):
    """Гистограмма, среднее (сплошная) и медиана (пунктир), легенда. Возвращает (fig, ax)."""
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    fig, ax = plt.subplots()
    ax.hist(values, bins=bins)
    ax.axvline(values.mean(), linestyle="-", label="среднее")
    ax.axvline(np.median(values), linestyle="--", label="медиана")
    ax.legend()
    return fig, ax


def lines_by_label(ax):
    return {line.get_label(): line for line in ax.lines}

fig, ax = hist_center(mass, 20)
assert len(ax.patches) == 20
lines = lines_by_label(ax)
assert set(lines) == {"среднее", "медиана"}, "нужны ровно две линии с этими подписями"
assert np.allclose(lines["среднее"].get_xdata(), 4201.754385964912)
assert np.allclose(lines["медиана"].get_xdata(), 4050.0)
assert lines["среднее"].get_linestyle() == "-"
assert lines["медиана"].get_linestyle() == "--"
assert set(texts(ax.get_legend().get_texts())) == {"среднее", "медиана"}
ax.set_xlabel("Масса, г")

fig2, ax2 = hist_center(np.array([1, 1, 1, 2, 10]), 5)
plt.close(fig2)
lines2 = lines_by_label(ax2)
assert np.allclose(lines2["среднее"].get_xdata(), 3)
assert np.allclose(lines2["медиана"].get_xdata(), 1)
print("Задача 7: ок")

### Задача 8. Исправить усечённую ось

Функция `misleading_bar` уже написана: она строит столбчатую диаграмму, у которой ось Y
начинается не с нуля. Выполните ячейку и посмотрите — различие между видами кажется огромным.

Напишите `fix_axis(ax)`, которая исправляет **уже построенную** диаграмму:

1. ось Y начинается с 0, а верхняя граница не ниже самого высокого столбца;
2. над каждым столбцом появляется его значение с одним знаком после запятой: `"190.0"`.

Данных функция не получает — только `ax`. Высоты столбцов берите с самого рисунка.

*Подсказка:* `ax.set_ylim(...)`, `ax.text(x, y, строка, ha="center", va="bottom")`.
Середина столбца `p` по горизонтали — `p.get_x() + p.get_width() / 2`.

In [ ]:
def misleading_bar(labels, values):
    """Столбчатая диаграмма с усечённой осью Y. Так делать не надо."""
    fig, ax = plt.subplots()
    ax.bar(labels, values, color="0.5")
    ax.set_ylim(min(values) - 2, max(values) + 2)
    return fig, ax


flipper_means = [190.0, 195.8, 217.2]      # средняя длина ласта по видам, мм
fig, ax = misleading_bar(SPECIES, flipper_means)
ax.set_title("Как было")
plt.show()

In [ ]:
def fix_axis(ax):
    """Ось Y от нуля, значения над столбцами. Ничего не возвращает."""
    ax.set_ylim(0, None)
    for i in ax.patches:
        height = i.get_height()
        x_center = i.get_x() + i.get_width() / 2
        ax.text(x_center, height, f"{height:.1f}", ha="center", va="bottom")

def label_y(t):
    # у ax.text позиция в get_position(), у подписей ax.bar_label — в t.xy
    return t.xy[1] if hasattr(t, "xy") else t.get_position()[1]

fig, ax = misleading_bar(SPECIES, flipper_means)
fix_axis(ax)
ax.set_title("Как стало")
assert ax.get_ylim()[0] == 0
assert ax.get_ylim()[1] >= 217.2
assert [p.get_height() for p in ax.patches] == flipper_means, "столбцы изменились"
assert sorted(texts(ax.texts)) == ["190.0", "195.8", "217.2"]
assert all(label_y(t) >= 190 for t in ax.texts), "подписи должны стоять над столбцами"

fig2, ax2 = misleading_bar(["a", "b"], [1500, 1520.25])
fix_axis(ax2)
plt.close(fig2)
assert ax2.get_ylim()[0] == 0 and ax2.get_ylim()[1] >= 1520.25
assert sorted(texts(ax2.texts)) == ["1500.0", "1520.2"]
print("Задача 8: ок")

### Задача 9. Ящики с усами по группам

По словарю «группа → значения» постройте ящики с усами, по ящику на группу, в порядке
словаря. Под каждым ящиком — название группы. Пропуски в данных возможны, и **каждый**
ящик должен быть построен по известным значениям своей группы.

```
fig, ax = box_groups(mass_by_species)
```

*Подсказка:* `ax.boxplot` принимает список массивов. Подписи ставьте отдельно:
`ax.set_xticks(позиции, labels=названия)` — ящики стоят на позициях 1, 2, 3, ...
(параметр `labels` у самого `boxplot` в новых версиях переименован, на него лучше не опираться).

In [ ]:
def box_groups(groups: dict):
    """Ящики с усами по группам, подписи групп по оси X. Возвращает (fig, ax)."""
    n = []
    k = []
    for name, values in groups.items():
        values = np.asarray(values, dtype=float)
        values = values[~np.isnan(values)]
        n.append(name)
        k.append(values)
    fig, ax = plt.subplots()
    ax.boxplot(k)
    ax.set_xticks(range(1, len(n) + 1), labels=n)
    return fig, ax
def has_median(ax, position, value):
    """Есть ли у ящика на позиции position горизонтальная черта на высоте value."""
    for line in ax.lines:
        xs, ys = np.asarray(line.get_xdata(), float), np.asarray(line.get_ydata(), float)
        if len(xs) == 2 and np.allclose(ys, value) and abs(xs.mean() - position) < 0.3:
            return True
    return False

fig, ax = box_groups(mass_by_species)
fig.canvas.draw()
assert list(ax.get_xticks()) == [1, 2, 3], "подписи должны стоять под ящиками: позиции 1, 2, 3"
assert texts(ax.get_xticklabels()) == SPECIES
assert has_median(ax, 1, 3700.0)     # Adelie
assert has_median(ax, 2, 3700.0)     # Chinstrap
assert has_median(ax, 3, 5000.0)     # Gentoo
ax.set_ylabel("Масса, г")

groups = {"б": np.array([3.0, 1.0, np.nan, 2.0]), "а": np.array([10.0, 20.0, 30.0, 40.0])}
fig2, ax2 = box_groups(groups)
fig2.canvas.draw()
plt.close(fig2)
assert texts(ax2.get_xticklabels()) == ["б", "а"]
assert has_median(ax2, 1, 2.0)
assert has_median(ax2, 2, 25.0)
print("Задача 9: ок")

### Задача 10. Группы на рассеянии без цвета

Постройте диаграмму рассеяния, где группы различаются **формой маркера**, а не цветом:
рисунок должен читаться в ч/б печати. На вход — словарь «группа → (x, y)».

Требования:

* по одному вызову `scatter` на группу, с подписью для легенды, легенда есть;
* маркеры берите по порядку из `MARKERS = ["o", "s", "^", "D", "v"]`, у разных групп они разные;
* все цвета (заливка и контур) — оттенки серого: чёрный, белый, `"0.5"` и т. п.;
* пары с пропуском в `x` или `y` не рисуются.

```
pairs = {s: (by_species("bill_length_mm")[s], by_species("flipper_length_mm")[s]) for s in SPECIES}
fig, ax = scatter_groups(pairs)
```

*Подсказка:* оттенок серого в matplotlib задаётся строкой с числом от `"0"` (чёрный)
до `"1"` (белый). Прозрачные маркеры с контуром — `facecolor="none", edgecolor="black"`.

In [ ]:
MARKERS = ["o", "s", "^", "D", "v"]
pairs = {s: (by_species("bill_length_mm")[s], by_species("flipper_length_mm")[s]) for s in SPECIES}

In [ ]:
def scatter_groups(pairs: dict):
    """Рассеяние по группам: разные маркеры, только серые цвета, легенда. Возвращает (fig, ax)."""
    fig, ax = plt.subplots()
    grays = ["gray", "0.3", "0.6", "0.8", "0.5"]
    for (name, (x, y)), marker, color in zip(pairs.items(), MARKERS, grays):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        good = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[good], y[good], marker=marker, color=color, label=name)
    ax.legend()
    return fig, ax

def is_grey(colors):
    colors = np.asarray(colors)
    return len(colors) == 0 or np.allclose(colors[:, 0], colors[:, 1]) and np.allclose(colors[:, 1], colors[:, 2])

def marker_shape(collection):
    return tuple(np.round(collection.get_paths()[0].vertices, 3).ravel())

fig, ax = scatter_groups(pairs)
colls = ax.collections
assert len(colls) == 3, "нужно по одному scatter на группу"
assert [c.get_label() for c in colls] == SPECIES
assert texts(ax.get_legend().get_texts()) == SPECIES
assert len({marker_shape(c) for c in colls}) == 3, "маркеры групп должны различаться"
assert all(is_grey(c.get_facecolors()) and is_grey(c.get_edgecolors()) for c in colls), "цвета должны быть серыми"
assert [len(c.get_offsets()) for c in colls] == [151, 68, 123]
ax.set_xlabel("Длина клюва, мм")
ax.set_ylabel("Длина ласта, мм")

fig2, ax2 = scatter_groups({"x": ([1, np.nan, 3], [1, 2, np.nan]), "y": ([1, 2], [3, 4])})
plt.close(fig2)
assert [len(c.get_offsets()) for c in ax2.collections] == [1, 2]
print("Задача 10: ок")

## Часть 3. Со звёздочкой

### Задача 11. Накопленная частота ★

Постройте график накопленной частоты: для каждого известного значения `v` — какая доля
наблюдений не превышает `v`. Отсортируйте известные значения по возрастанию:
`x[0] <= x[1] <= ... <= x[n-1]`. График проходит через точки `(x[i], (i + 1) / n)`
и рисуется **ступеньками**: между соседними значениями доля не меняется.
Ось Y — от 0 до 1 включительно.

```
fig, ax = ecdf(mass)
```

Когда тесты пройдут, найдите по рисунку, какая доля пингвинов весит не больше 4000 г,
и проверьте себя вычислением.

*Подсказка:* ступенчатую линию рисует `ax.step(x, y, where="post")`.

In [ ]:
def ecdf(values):
    """Ступенчатый график накопленной частоты по известным значениям. Возвращает (fig, ax)."""
    # ваш код здесь
    pass


fig, ax = ecdf(mass)
assert len(ax.lines) == 1
line = ax.lines[0]
x, y = np.asarray(line.get_xdata()), np.asarray(line.get_ydata())
assert len(x) == 342
assert np.all(np.diff(x) >= 0), "значения должны идти по возрастанию"
assert x[0] == 2700 and x[-1] == 6300
assert np.isclose(y[0], 1 / 342) and y[-1] == 1
assert line.get_drawstyle() == "steps-post", "нужны ступеньки: step(..., where='post')"
assert ax.get_ylim()[0] == 0 and ax.get_ylim()[1] >= 1
ax.axvline(4000, color="0.6", linestyle=":")
ax.set_xlabel("Масса, г")

fig2, ax2 = ecdf(np.array([3.0, 1.0, np.nan, 2.0, 2.0]))
plt.close(fig2)
assert list(ax2.lines[0].get_xdata()) == [1, 2, 2, 3]
assert list(ax2.lines[0].get_ydata()) == [0.25, 0.5, 0.75, 1.0]
print("Задача 11: ок")

### Задача 12. Тепловая карта с подписями ★

Постройте тепловую карту таблицы чисел в оттенках серого (`cmap="Greys"`) со шкалой
соответствия цвета и значения. Подпишите строки и столбцы. В центре каждой клетки
напишите её значение целым числом; текст на тёмных клетках — белый, на светлых — чёрный.
Тёмной считается клетка, значение которой **строго больше** середины между минимумом
и максимумом таблицы.

```
fig, ax = heatmap(CROSS, SPECIES, ISLANDS)
```

`CROSS` — таблица сопряжённости «вид × остров» из задачи 4.4 прошлого практикума.

*Подсказка:* `ax.imshow(matrix, cmap=...)`, шкала — `fig.colorbar(image, ax=ax)`.
Клетка в строке `i` и столбце `j` имеет центр в точке `x = j`, `y = i`.

In [ ]:
ISLANDS = ["Biscoe", "Dream", "Torgersen"]
CROSS = pd.crosstab(penguins["species"], penguins["island"]).loc[SPECIES, ISLANDS].to_numpy()
CROSS

In [ ]:
def heatmap(matrix, row_labels, col_labels):
    """Тепловая карта в сером с подписями клеток и шкалой. Возвращает (fig, ax)."""
    # ваш код здесь
    pass


from matplotlib.colors import to_rgb

def cell_texts(ax):
    """Словарь: (строка, столбец) -> (текст, цвет)."""
    result = {}
    for t in ax.texts:
        x, y = t.get_position()
        result[(round(y), round(x))] = (t.get_text(), to_rgb(t.get_color()))
    return result

WHITE, BLACK = (1.0, 1.0, 1.0), (0.0, 0.0, 0.0)

fig, ax = heatmap(CROSS, SPECIES, ISLANDS)
assert len(ax.images) == 1
assert np.array_equal(ax.images[0].get_array(), CROSS)
assert ax.images[0].get_cmap().name == "Greys"
assert len(fig.axes) == 2, "нет шкалы (colorbar)"
fig.canvas.draw()
assert texts(ax.get_xticklabels()) == ISLANDS
assert texts(ax.get_yticklabels()) == SPECIES
cells = cell_texts(ax)
assert len(ax.texts) == 9 and len(cells) == 9
assert cells[(0, 0)] == ("44", BLACK)
assert cells[(1, 1)] == ("68", WHITE)
assert cells[(2, 0)] == ("124", WHITE)
assert cells[(1, 0)] == ("0", BLACK)

fig2, ax2 = heatmap(np.array([[1, 2, 3], [4, 5, 6]]), ["a", "b"], ["x", "y", "z"])
plt.close(fig2)
cells2 = cell_texts(ax2)
assert [cells2[(i, j)][0] for i in range(2) for j in range(3)] == ["1", "2", "3", "4", "5", "6"]
assert [cells2[(i, j)][1] == WHITE for i in range(2) for j in range(3)] == [False] * 3 + [True] * 3

fig3, ax3 = heatmap(np.array([[7, 7]]), ["r"], ["p", "q"])
plt.close(fig3)
assert all(color == BLACK for text, color in cell_texts(ax3).values())
print("Задача 12: ок")